In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CRDS_SERVER_URL']='https://jwst-crds.stsci.edu' ## download JWST reference file
os.environ["CRDS_CONTEXT"] = "jwst_1364.pmap"
os.environ['CRDS_PATH']='/home/zhongyi/CRDS'
import json
import matplotlib.pyplot as plt
from glob import glob
from astropy.io import fits
from astropy.stats import sigma_clipped_stats
from astropy.visualization import simple_norm
from remstriping import striping_noise
import warnings 
warnings.filterwarnings("ignore")
input_dir = "/Data/JWST_ZY/f200w/stage1"
output_dir = "/Data/JWST_ZY/f200w/stage1"

In [ ]:
def show(filename, threshold = "NAN"):
    print(f"Plot {filename}")
    with fits.open(filename) as hdul:
        sci = hdul["SCI"].data
        # sci = hdul[1].data
    _, median, std = sigma_clipped_stats(sci)
    norm = simple_norm(sci, vmin = median-2*std, vmax = median+2*std)
    fig, ax = plt.subplots(figsize = (6,6))
    ax.imshow(sci, norm = norm, cmap = "jet", origin = "lower")
    ax.set_title(f"{os.path.basename(filename)}, threshold = {threshold}")
    plt.show()

In [ ]:
def run_1overf(ratefile, threshold):
    print("Doing 1 over noise subtraction for {}".format(ratefile))
    pre_name = ratefile.replace("rate.fits", "rate_pre1f.fits")
    sn = striping_noise(INPUTDIR=input_dir, OUTPUTDIR=output_dir, MASKTHRESH=threshold)
    sn.measure_striping(ratefile, origfilename = pre_name, save_patterns = False, apply_flat = True)
    print("end")

In [ ]:
def prompt_threshold():
    while True:
        s = input("请输入新的阈值 (0-1): ")
        try:
            t = float(s)
        except ValueError:
            print("不是合法浮点数，请重试")
            continue
        if not (0 <= t <= 1):
            print("必须在 0-1 范围内，请重试")
            continue
        return t

In [ ]:
def creat_json(input_dir = None, output_file = None):

    syntax = "syntax: creat_json(input_dir = , output_dir = )"
    if None in [input_dir, output_file]:
        print(syntax)
        return

    files = sorted(glob(os.path.join(input_dir, "*rate.fits")))
    files_base = [os.path.basename(file).split("_rate.fits")[0] for file in files]
    file_no_suffix = [os.path.join(input_dir, base) for base in files_base]
    dic = {file: [0, 0] for file in file_no_suffix} 
    for file, x_list in dic.items():
            x_list[1] = 0.8
    with open(output_file, 'w') as j:
        json.dump(dic, j, indent = 4)

    print("json file has been created.")

# creat_json(input_dir = "/Data/JWST_ZY/f200w/stage1", output_file = "json/changhao_f200w.json")

In [ ]:
def main():

    ii = 0
    with open("json/changhao_f200w.json", 'r') as j:
        needs = json.load(j)
    to_process = [k for k, v in needs.items() if v[0] == 0]

    to_process = sorted(to_process)
    for basename in to_process:

        print(f"Adjusting threshold for {basename}_rate.fits")

        prename = os.path.join(input_dir, f"{basename}_rate_pre1f.fits")
        ratefile = os.path.join(input_dir, f"{basename}_rate.fits")
        threshold = float(needs[basename][1])

        while True:

            show(ratefile, threshold)
            cmd = input("效果如何？(y = 好, n = 还要调): ").strip().lower()

            if cmd == "y":
                print("OK, 下一张")
                needs[basename][0] = 1
                needs[basename][1] = threshold
                break

            if cmd == "n":
                print("再调阈值")
                threshold = prompt_threshold()
                #删掉文件并用原本的文件替代
                if os.path.isfile(ratefile):
                    print(f"Delete original ratefile {ratefile}")
                    os.remove(ratefile)
                os.rename(prename, ratefile)
                #对原始ratefile做1overf
                run_1overf(ratefile, threshold, )
                
            if cmd == "break":
                break


        ii += 1
        print(f"这是第{ii}张图")
        if cmd == "break":
            print("先出来")
            break



    with open("json/changhao_f200w.json", "w") as j:
        json.dump(needs, j, indent=4, ensure_ascii=False)
    

    print("更新完成")


In [ ]:
if __name__ == "__main__":
    main()